# QLoRA Fine-Tuning: Qwen/Qwen3-8B on Fable-5 Traces (Unsloth)

- **Model:** Qwen/Qwen3-8B
- **Dataset:** Glint-Research/Fable-5-traces (pi_agent)
- **Quant:** INT2 (GGUF Q2_K)
- **Target Size:** ~2.0 GB
- **Output:** ./model/qwen3-8b-int2
- **Hardware:** Kaggle 2x T4 (16 GB each)
- **Framework:** Unsloth


## 1. Setup


In [ ]:
!pip install -q unsloth datasets einops sentencepiece tiktoken
!pip install -q llama-cpp-python


In [ ]:
import os, gc, json, torch, matplotlib.pyplot as plt
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments


In [ ]:
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  VRAM: {p.total_memory / 1024**3:.2f} GB")


## 2. Dataset


In [ ]:
ds = load_dataset("Glint-Research/Fable-5-traces", "pi_agent", split="train")
print(f"Rows: {len(ds)}  Columns: {ds.column_names}")
print("\nFirst row keys:")
for k, v in ds[0].items():
    if k == "messages":
        print(f"  messages: list[{len(v)}]")
    else:
        print(f"  {k}: {type(v).__name__} = {str(v)[:200]}")


In [ ]:
import json
def flatten_content(parts):
    if isinstance(parts, str):
        return parts
    result = []
    for part in parts:
        t = part["type"]
        if t == "thinking":
            result.append(f"<thinking>\n{part['content']}\n</thinking>")
        elif t == "text":
            result.append(part["text"])
        elif t == "toolCall":
            tool = part["toolUse"]
            inp = json.dumps(tool["input"], indent=2) if isinstance(tool["input"], dict) else str(tool["input"])
            result.append(f"<tool_call>\n<tool_name>{tool['name']}</tool_name>\n<input>\n{inp}\n</input>\n</tool_call>")
    return "\n".join(result)
def reformat_row(row):
    messages = []
    for m in row["messages"]:
        messages.append({"role": m["role"], "content": flatten_content(m["content"])})
    return {"messages": messages}
ds = ds.map(reformat_row, remove_columns=ds.column_names)
print(f"Reformatted: {len(ds)} rows")
print("\nSample:")
for i, m in enumerate(ds[0]["messages"][:3]):
    c = m["content"][:250].replace("\n", " ")
    print(f"  [{i}] {m['role']}: {c}")


## 3. QLoRA Fine-Tuning (Unsloth)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3-8B",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
MODEL_NAME = "Qwen/Qwen3-8B"
print("Model loaded via Unsloth")


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print(f"Trainable: {model.num_parameters(only_trainable=True):,} / {model.num_parameters():,}")


In [ ]:
args = TrainingArguments(
    output_dir="./ckpt-qwen3-8b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True, bf16=False,
    optim="adamw_8bit",
    logging_steps=1, log_level="info", logging_first_step=True,
    save_strategy="no", report_to="none",
    warmup_ratio=0.03, lr_scheduler_type="cosine",
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
)


In [ ]:
def fmt(example):
    return tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
trainer = SFTTrainer(
    model=model, args=args, train_dataset=ds,
    formatting_func=fmt, tokenizer=tokenizer, max_seq_length=4096,
)


In [ ]:
trainer.train()


In [ ]:
logs = trainer.state.log_history
steps = [x["step"] for x in logs if "loss" in x]
losses = [x["loss"] for x in logs if "loss" in x]
plt.figure(figsize=(10,5))
plt.plot(steps, losses)
plt.xlabel("Step"); plt.ylabel("Loss")
plt.title(f"Training Loss - {MODEL_NAME}")
plt.grid(True); plt.show()


## 4. Merge + Quantize (Unsloth built-in)


In [ ]:
def get_dir_size(path):
    return sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fn in os.walk(path) for f in fn) / (1024**3)
def get_file_size(path):
    return os.path.getsize(path) / (1024**3)


In [ ]:
OUT = "./model/qwen3-8b-int2"
os.makedirs(OUT, exist_ok=True)
print(f"Saving to {OUT}")


In [ ]:
print("Saving GGUF Q2_K via Unsloth...")
model.save_pretrained_gguf(OUT, tokenizer, quantization_method="q2_k")
gguf_files = [f for f in os.listdir(OUT) if f.endswith(".gguf")]
if gguf_files:
    GGUF_PATH = os.path.join(OUT, gguf_files[0])
    sz = get_file_size(GGUF_PATH)
    print(f"GGUF Q2_K size: {sz:.2f} GB (target 2.0 GB)")
    if abs(sz - 2.0) > 0.2:
        print(f"WARNING: deviates >0.2 GB from target")
    else:
        print("OK: within tolerance")
else:
    print("WARNING: No .gguf file found in output directory")


## 5. Save Confirmation


In [ ]:
print(f"Model saved to: {OUT}")
print(f"Contents: {os.listdir(OUT)}")


## 6. Quick Sanity Eval


In [ ]:
from llama_cpp import Llama
gguf_files = [f for f in os.listdir(OUT) if f.endswith(".gguf")]
assert gguf_files, "No GGUF file found"
GGUF_PATH = os.path.join(OUT, gguf_files[0])
llm = Llama(model_path=GGUF_PATH, n_ctx=4096, n_gpu_layers=-1, verbose=False)
prompts = [
    "Write a Python function to sort dicts by a key.",
    "Diff between git merge and rebase?",
    "Bash one-liner: files > 100MB.",
    "Implement LRU cache in Python.",
    "CORS in Flask?",
]
for i, p in enumerate(prompts):
    r = llm(p, max_tokens=256, temperature=0.7, echo=False)
    print(f"\n=== {i+1} ===\nQ: {p}\nA: {r['choices'][0]['text'].strip()}")
